In [2]:
import scipy.sparse as sp
import numpy as np
import matplotlib.pyplot as plt
from ldpc import BpOsdDecoder
from ldpc.mod2 import rank
from ldpc.codes import rep_code
#from ldpc.bpa import hgp

sqrt_pi = np.sqrt(np.pi)

In [ ]:
def construct_hgp_stabilizers(H1, H2):
    """
    Build hypergraph stabilisers hz and hz 
    same pattern as when did ages ago
    """
    H1 = sp.csc_matrix(H1)
    H2 = sp.csc_matrix(H2)
    
    r1, n1 = H1.shape
    r2, n2 = H2.shape
    
    I_n1 = sp.eye(n1, format='csc')
    I_n2 = sp.eye(n2, format='csc')
    I_r1 = sp.eye(r1, format='csc')
    I_r2 = sp.eye(r2, format='csc')
    
    # HX = [ H1 tensor  I_n2 ,  I_r1 tensor H2.T ]
    HX_left = sp.kron(H1, I_n2, format='csc')
    HX_right = sp.kron(I_r1, H2.T, format='csc')
    Hx = sp.hstack([HX_left, HX_right], format='csc')
    
    # HZ = [ I_n1 tensor H2 ,  H1.T tensor I_r2 ]
    HZ_left = sp.kron(I_n1, H2, format='csc')
    HZ_right = sp.kron(H1.T, I_r2, format='csc')
    Hz = sp.hstack([HZ_left, HZ_right], format='csc')
    
    return Hx, Hz

Hx, Hz = construct_hgp_stabilizers(H1, H2)

In [3]:
def gkp_error_and_probability(sigma, n, cutoff=5):
    """error and probability returned"""
    displacement = np.random.normal(0, sigma, n)

    residual = (
        (displacement + sqrt_pi / 2) % sqrt_pi
    ) - sqrt_pi / 2

    error = (
        np.rint((displacement - residual) / sqrt_pi).astype(np.uint8) % 2
    )
    #gaussian prob
    k = np.arange(-cutoff, cutoff + 1)
    likelihood = np.exp(
        -(residual[:, None] + k[None, :] * sqrt_pi) ** 2
        / (2 * sigma**2)
    )

    odd = (np.abs(k) % 2) == 1
    p_error = likelihood[:, odd].sum(axis=1) / likelihood.sum(axis=1)
    #for bp
    p_error = np.clip(p_error, 1e-12, 1 - 1e-12)

    return error, p_error